In [1]:
!pip install datasets
from datasets import load_dataset
import pandas as pd, json, ast

ds = load_dataset("ai4privacy/pii-masking-200k", split="train")
raw = ds.to_pandas()

print(raw.shape)
print(raw.columns.tolist())
raw.head(3)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\Victus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(209261, 9)
['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']


,source_text,target_text,privacy_mask,span_labels,mbert_text_tokens,mbert_bio_labels,id,language,set
0,A student's assessment was found on device bea...,A student's assessment was found on device bea...,"[{'value': '06-184755-866851-3', 'start': 57, ...","[[0, 57, ""O""], [57, 75, ""PHONEIMEI""], [75, 138...","[A, student, ', s, assessment, was, found, on,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-P...",165761,en,train
1,"Dear Omer, as per our records, your license 78...","Dear [FIRSTNAME], as per our records, your lic...","[{'value': 'Omer', 'start': 5, 'end': 9, 'labe...","[[0, 5, ""O""], [5, 9, ""FIRSTNAME""], [9, 44, ""O""...","[Dear, Omer, ,, as, per, our, records, ,, your...","[O, B-FIRSTNAME, O, O, O, O, O, O, O, O, B-VEH...",165762,en,train
2,Kattie could you please share your recomndatio...,[FIRSTNAME] could you please share your recomn...,"[{'value': 'Kattie', 'start': 0, 'end': 6, 'la...","[[0, 6, ""FIRSTNAME""], [6, 75, ""O""], [75, 77, ""...","[Kat, ##tie, could, you, pl, ##eas, ##e, share...","[B-FIRSTNAME, I-FIRSTNAME, O, O, O, O, O, O, O...",165763,en,train


In [2]:
def parse_mask(m):
    if isinstance(m, str):
        try: return json.loads(m)
        except: return ast.literal_eval(m)
    return m

raw["mask"] = raw["privacy_mask"].apply(parse_mask)
all_labels = set(item["label"] for m in raw["mask"] for item in m)
print(sorted(all_labels))

['ACCOUNTNAME', 'ACCOUNTNUMBER', 'AGE', 'AMOUNT', 'BIC', 'BITCOINADDRESS', 'BUILDINGNUMBER', 'CITY', 'COMPANYNAME', 'COUNTY', 'CREDITCARDCVV', 'CREDITCARDISSUER', 'CREDITCARDNUMBER', 'CURRENCY', 'CURRENCYCODE', 'CURRENCYNAME', 'CURRENCYSYMBOL', 'DATE', 'DOB', 'EMAIL', 'ETHEREUMADDRESS', 'EYECOLOR', 'FIRSTNAME', 'GENDER', 'HEIGHT', 'IBAN', 'IP', 'IPV4', 'IPV6', 'JOBAREA', 'JOBTITLE', 'JOBTYPE', 'LASTNAME', 'LITECOINADDRESS', 'MAC', 'MASKEDNUMBER', 'MIDDLENAME', 'NEARBYGPSCOORDINATE', 'ORDINALDIRECTION', 'PASSWORD', 'PHONEIMEI', 'PHONENUMBER', 'PIN', 'PREFIX', 'SECONDARYADDRESS', 'SEX', 'SSN', 'STATE', 'STREET', 'TIME', 'URL', 'USERAGENT', 'USERNAME', 'VEHICLEVIN', 'VEHICLEVRM', 'ZIPCODE']


In [3]:
if "language" in raw.columns:
    raw = raw[raw["language"].str.lower().isin(["en", "english"])]

TARGET = {"EMAIL": "email", "PHONENUMBER": "phone", "PASSWORD": "credential"}

def window_bounds(text, s, e, pad=30):
    a, b = max(0, s - pad), min(len(text), e + pad)
    while 0 < a < s and not text[a-1].isspace():      # start on a word boundary
        a += 1
    while e < b < len(text) and not text[b].isspace():  # end on a word boundary
        b -= 1
    return a, b

rows = []
for _, r in raw.iterrows():
    text, items = r["source_text"], r["mask"]
    for item in items:
        lab = TARGET.get(item["label"].upper())
        if lab is None:
            continue
        a, b = window_bounds(text, item["start"], item["end"])
        clash = any(
            o is not item
            and TARGET.get(o["label"].upper()) not in (None, lab)
            and o["start"] < b and o["end"] > a
            for o in items
        )
        if clash:
            continue
        rows.append({"text": text[a:b].strip(), "label": lab, "source": "ai4privacy"})

pub = pd.DataFrame(rows).drop_duplicates(subset="text")
print(pub["label"].value_counts())

label
email         3584
credential    2594
phone         2229
Name: count, dtype: int64


In [4]:
# if "language" in raw.columns:
#     raw = raw[raw["language"].str.lower().isin(["en", "english"])]

# def window(text, start, end, pad=40):
#     return text[max(0, start - pad): end + pad].strip()

# rows = []
# for _, r in raw.iterrows():
#     text = r["source_text"]
#     for item in r["mask"]:
#         lab = item["label"].upper()
#         if lab == "EMAIL":
#             label = "email"
#         elif lab == "PHONENUMBER":
#             label = "phone"
#         elif lab == "PASSWORD":
#             label = "credential"
#         else:
#             continue
#         rows.append({
#             "text": window(text, item["start"], item["end"]),
#             "label": label,
#             "source": "ai4privacy"
#         })

# pub = pd.DataFrame(rows).drop_duplicates(subset="text")
# print(pub["label"].value_counts())

In [5]:
pub = pd.concat(
    [g.sample(min(len(g), 1500), random_state=42) for _, g in pub.groupby("label")]
).reset_index(drop=True)

pub.to_csv("../data/ai4privacy.csv", index=False)
print(pub.shape)
print(pub["label"].value_counts())
pub.groupby("label").sample(4, random_state=1)

(4500, 3)
label
credential    1500
email         1500
phone         1500
Name: count, dtype: int64


,text,label,source
91,access to Kennith.Maggio with X8VYiiUD_Fge. Pl...,credential,ai4privacy
75,password currently set as ZF4KA1WWZFoM for wor...,credential,ai4privacy
1264,"doc, please use this password jy04bGROxP5T. Le...",credential,ai4privacy
330,to create a unique password 8k6risptDPWd to ac...,credential,ai4privacy
2382,sent a receipt to your email: Heidi40@yahoo.com.,email,ai4privacy
1784,Miss Mireya Addison Ledner at Justen_Stracke75...,email,ai4privacy
1882,with this account is Lysanne_Schuppe@gmail.com.,email,ai4privacy
2680,email legal analysis links to Iliana.Halvorson...,email,ai4privacy
3042,"For inquiries, contact us via +53.946 271 3676.",phone,ai4privacy
3133,in advance to 54706388. Dial 08773 447730 for ...,phone,ai4privacy


# 02 - Load Public Dataset (ai4privacy)

## Purpose
Add a second, independent dataset with realistic human-style emails, phone numbers and
passwords, so the project uses two different data sources.

## Input / Output
- **Input:** `ai4privacy/pii-masking-200k` from Hugging Face
- **Output:** `data/ai4privacy.csv` (4,500 rows: 1,500 each of email, phone, credential)

## What we did
1. Loaded the dataset with the `datasets` library and inspected its columns
   (`source_text`, `privacy_mask`, `language`).
2. Parsed the `privacy_mask` column, which lists each PII value with its label and
   position, and printed all 56 label types.
3. Kept English rows only.
4. Mapped three labels to our format: EMAIL -> email, PHONENUMBER -> phone,
   PASSWORD -> credential.
5. Cut a 40-character window around each value, so the text looks like a short OCR
   line instead of a long paragraph.
6. Removed duplicates, capped each class at 1,500 rows, and saved the CSV.

## Key concepts learned
- Public datasets rarely match your format, so they need **conversion**.
- Raw counts were email 4092, credential 2622, phone 2425. Capping at 1,500 avoids one
  class dominating.
- Always look at the real label list before filtering.

## Problems and fixes
- **Phone label bug:** matching "PHONE" also caught PHONEIMEI (device IDs). Fixed by
  matching the exact label PHONENUMBER.
- **pandas error:** `groupby().apply()` dropped the label column in new pandas versions
  (KeyError: 'label'). Fixed by sampling each group inside `pd.concat`.
- Hugging Face short names stopped working, so full `owner/name` is needed.

## Brief requirement covered
Second dataset, data source and types description, preprocessing of inconsistencies.

## Limitation to mention in the report
Windows are cut mid-sentence and look different from Faker's short templates, so a model
could partly learn the source style. We check this per source and with real screenshots.

- **Ambiguous windows:** error analysis in notebook 06 showed that most validation errors
  came from ai4privacy windows containing two PII types (for example an email next to a
  password), so the true label was unclear. We reduced the window to 30 characters, snapped
  it to whole words, and removed windows containing a different PII type.